**PROYECTO:**
**Modelado espacial para estimar la susceptibilidad a inundaciones usando algoritmos de aprendizaje automatico con datos geomorfologicos**

Autor: Sebastian Velosa Bohorquez, Curso: Percepción Remota Avanzada,

**NOTEBOOK 06/07**

# Preparación de muestras para entrenamiento, validacion y prueba

**Objetivo:**

Este notebook verifica cuatro GeoTIFF, identifica píxeles válidos comunes, extrae una muestra balanceada de 5.000 píxeles y exporta un CSV por cada DEM, tres en total. Los tres CSV comparten exactamente las mismas ubicaciones y la misma partición preliminar train/validation/test.

**Requisitos de entrada:** tres GeoTIFF multibanda de 9 bandas (FABDEM, Copernicus y ANADEM) y un GeoTIFF de inventario binario (0 = no inundado, 1 = inundado). Todos deben estar previamente alineados en EPSG:31972, 30 m, con idénticas dimensiones, extensión y transformación de píxel.

**Herramientas:** Google Earth Engine

**Contenido:**

1. Instalación de Librerias
2. Configuración de rutas
3. Comprobar archivos, bandas y alineación espacial
4. Encontrar píxeles válidos comunes
5. Extraer 5.000 muestras balanceadas y asignar train/validation/test
6. Extraer las nueve variables de cada DEM y exportar CSV
7. Verificación final de los archivos exportados

**Resultados:**


Archivos almacenados en la carpeta 04_sampling

*   muestras_copernicus_5000.csv:	Nueve variables de Copernicus y etiquetas
*   muestras_anadem_5000.csv	:Nueve variables de ANADEM y etiquetas
*   muestras_fabdem_5000.csv	:Nueve variables de FABDEM y etiquetas
*   ubicaciones_muestra_particiones.csv	:Ubicaciones, etiquetas y particiones comunes









## 1. Instalación de Librerias

In [ ]:
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from sklearn.model_selection import train_test_split

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Configuración de rutas

Se importan los archivos `.tiff `creados en el notebook 03. Estos son tres rásteres multibanda de nueve variables y un ráster binario de inventario de inundaciones. Cuentan con las 9 variables topograficas e hidrograficas.

Stack creado: `/content/resultados_variables/Stack_topografico_hidrologico.tif`

Orden de bandas: ['Elevation', 'Slope', 'HAND', 'FlowAccumulation', 'DistRiver', 'TWI', 'Curvature', 'TPI', 'TRI']

In [ ]:
# Reemplaza estas rutas por las ubicaciones reales de los archivos en Drive.
# ajustar únicamente las rutas y, si hace falta, los códigos de clase de inundacion

DEM_PATHS = {
    'FABDEM': '/content/drive/MyDrive/Sinu_Flood_Susceptibility/03_data_processed/VARIABLES_TOPOGRAFICAS/Stack_topografico_hidrologico_FABDEM.tif',
    'Copernicus': '/content/drive/MyDrive/Sinu_Flood_Susceptibility/03_data_processed/VARIABLES_TOPOGRAFICAS/Stack_topografico_hidrologico_COP.tif',
    'ANADEM': '/content/drive/MyDrive/Sinu_Flood_Susceptibility/03_data_processed/VARIABLES_TOPOGRAFICAS/Stack_topografico_hidrologico_ANADEM.tif'
}
INVENTORY_PATH = '/content/drive/MyDrive/Sinu_Flood_Susceptibility/03_data_processed/FLOOD/FloodInventory_v1.tif'
OUTPUT_DIR = '/content/drive/MyDrive/Sinu_Flood_Susceptibility/04_samples'

# Orden esperado de las 9 bandas en CADA GeoTIFF multibanda.
BAND_NAMES = [

    'Elevation', 'Slope', 'HAND', 'FlowAccumulation', 'DistRiver', 'TWI', 'Curvature', 'TPI', 'TRI'
]

# Parametros de muestras

N_SAMPLES = 5000
CLASS_NO_FLOOD = 0
CLASS_FLOOD = 1
SEED = 42
CRS_EXPECTED = 'EPSG:31972'
PIXEL_SIZE = 30  # metros
# Valores que pueden representar NoData aunque no estén declarados en los metadatos.
EXTRA_NODATA_VALUES = [-9999]

# Partición preliminar compartida por los tres modelos.
TEST_SIZE = 0.20
VALIDATION_SIZE_OF_REMAINDER = 0.25  # 25% del 80% restante = 20% del total

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Carpeta de salida:', OUTPUT_DIR)


Carpeta de salida: /content/drive/MyDrive/Sinu_Flood_Susceptibility/04_samples


## 3. Comprobar archivos, bandas y alineación espacial

In [ ]:
all_paths = list(DEM_PATHS.values()) + [INVENTORY_PATH]
missing = [p for p in all_paths if not os.path.isfile(p)]
if missing:
    raise FileNotFoundError('No se encontraron estos archivos. Corrige las rutas:\\n' + '\\n'.join(missing))

profiles = {}
for name, path in DEM_PATHS.items():
    with rasterio.open(path) as src:
        profiles[name] = {
            'crs': src.crs,
            'transform': src.transform,
            'width': src.width,
            'height': src.height,
            'count': src.count,
            'bounds': src.bounds,
            'res': src.res,
            'nodata': src.nodata,
        }
        print(f"{name}: bandas={src.count}, tamaño={src.width} x {src.height}, CRS={src.crs}, resolución={src.res}, NoData={src.nodata}")

with rasterio.open(INVENTORY_PATH) as src:
    inv_profile = {
        'crs': src.crs, 'transform': src.transform,
        'width': src.width, 'height': src.height,
        'count': src.count, 'bounds': src.bounds,
        'res': src.res, 'nodata': src.nodata
    }
    print(f"INVENTARIO: bandas={src.count}, tamaño={src.width} x {src.height}, CRS={src.crs}, resolución={src.res}, NoData={src.nodata}")

reference = profiles['FABDEM']
errors = []
for name, p in profiles.items():
    if p['count'] != len(BAND_NAMES): errors.append(f"{name}: se esperaban 9 bandas y tiene {p['count']}.")
    if p['crs'] is None or p['crs'].to_string() != CRS_EXPECTED: errors.append(f'{name}: CRS distinto de {CRS_EXPECTED}.')
    if (p['width'], p['height']) != (reference['width'], reference['height']): errors.append(f'{name}: dimensiones diferentes de FABDEM.')
    if p['transform'] != reference['transform']: errors.append(f'{name}: transformación/ubicación de píxel diferente de FABDEM.')
    if p['res'] != (PIXEL_SIZE, PIXEL_SIZE): errors.append(f'{name}: resolución distinta de 30 x 30 m.')

for key in ['crs', 'width', 'height', 'transform', 'res']:
    if inv_profile[key] != reference[key]: errors.append(f'Inventario: {key} no coincide con FABDEM.')
if inv_profile['count'] != 1: errors.append('El inventario debe tener una sola banda.')
if errors:
    raise ValueError('Hay problemas de alineación. No continúes hasta corregirlos:\\n- ' + '\\n- '.join(errors))

print('\\nVerificación espacial superada: los cuatro rásteres comparten grilla.')


FABDEM: bandas=9, tamaño=3103 x 3848, CRS=EPSG:31972, resolución=(30.0, 30.0), NoData=-9999.0
Copernicus: bandas=9, tamaño=3103 x 3848, CRS=EPSG:31972, resolución=(30.0, 30.0), NoData=-9999.0
ANADEM: bandas=9, tamaño=3103 x 3848, CRS=EPSG:31972, resolución=(30.0, 30.0), NoData=-9999.0
INVENTARIO: bandas=1, tamaño=3103 x 3848, CRS=EPSG:31972, resolución=(30.0, 30.0), NoData=None
\nVerificación espacial superada: los cuatro rásteres comparten grilla.


## 4. Encontrar píxeles válidos comunes

Se revisan los archivos por bloques para no cargar todos los rásteres completos en memoria. Un píxel solo es elegible si las 27 variables de los tres DEM y la etiqueta del inventario son válidas.

In [ ]:
candidates = {CLASS_NO_FLOOD: [], CLASS_FLOOD: []}
dem_sources = {name: rasterio.open(path) for name, path in DEM_PATHS.items()}

try:
    with rasterio.open(INVENTORY_PATH) as inv:
        width = inv.width
        for _, window in inv.block_windows(1):
            labels = inv.read(1, window=window)
            valid = inv.read_masks(1, window=window) > 0
            valid &= np.isin(labels, [CLASS_NO_FLOOD, CLASS_FLOOD])

            for name, src in dem_sources.items():
                data = src.read(window=window)
                valid &= (src.read_masks(window=window) > 0).all(axis=0)
                valid &= np.isfinite(data).all(axis=0)
                for nodata_value in EXTRA_NODATA_VALUES:
                    valid &= (data != nodata_value).all(axis=0)

            row0, col0 = int(window.row_off), int(window.col_off)
            for cls in [CLASS_NO_FLOOD, CLASS_FLOOD]:
                rr, cc = np.where(valid & (labels == cls))
                if len(rr):
                    flat_idx = (rr + row0) * width + (cc + col0)
                    candidates[cls].append(flat_idx.astype(np.int64))
finally:
    for src in dem_sources.values():
        src.close()

candidates = {
    cls: np.concatenate(chunks) if chunks else np.array([], dtype=np.int64)
    for cls, chunks in candidates.items()
}
counts = {int(cls): int(len(idx)) for cls, idx in candidates.items()}
print('Píxeles válidos comunes por clase:', counts)

n_per_class = N_SAMPLES // 2
if N_SAMPLES % 2 != 0:
    raise ValueError('N_SAMPLES debe ser par para un balance 50/50.')
for cls, count in counts.items():
    if count < n_per_class:
        raise ValueError(f'La clase {cls} solo tiene {count} píxeles válidos; se necesitan {n_per_class}.')
print('Hay suficientes píxeles para extraer la muestra balanceada.')


Píxeles válidos comunes por clase: {0: 5980047, 1: 481763}
Hay suficientes píxeles para extraer la muestra balanceada.


## 5. Extraer 5.000 muestras balanceadas y asignar train/validation/test

In [ ]:
rng = np.random.default_rng(SEED)
selected = []
for cls in [CLASS_NO_FLOOD, CLASS_FLOOD]:
    selected.append(rng.choice(candidates[cls], size=n_per_class, replace=False))
selected = np.concatenate(selected)
rng.shuffle(selected)

height = reference['height']
width = reference['width']
rows = selected // width
cols = selected % width
with rasterio.open(DEM_PATHS['FABDEM']) as src:
    coords = [src.transform * (int(c) + 0.5, int(r) + 0.5) for r, c in zip(rows, cols)]
    xs = np.array([xy[0] for xy in coords])
    ys = np.array([xy[1] for xy in coords])

with rasterio.open(INVENTORY_PATH) as inv:
    labels = inv.sample(coords, indexes=[1])
    y = np.array([int(v[0]) for v in labels], dtype=np.uint8)

samples = pd.DataFrame({
    'sample_id': [f'S{i:05d}' for i in range(1, N_SAMPLES + 1)],
    'row': rows.astype(np.int32),
    'col': cols.astype(np.int32),
    'x': xs,
    'y_coord': ys,
    'Flood': y
})

# Partición estratificada única, reutilizada por los tres DEM.
train_df, test_df = train_test_split(
    samples, test_size=TEST_SIZE, random_state=SEED, stratify=samples['Flood']
)
train_df, val_df = train_test_split(
    train_df, test_size=VALIDATION_SIZE_OF_REMAINDER,
    random_state=SEED, stratify=train_df['Flood']
)
split_lookup = {}
split_lookup.update({sid: 'train' for sid in train_df['sample_id']})
split_lookup.update({sid: 'validation' for sid in val_df['sample_id']})
split_lookup.update({sid: 'test' for sid in test_df['sample_id']})
samples['split'] = samples['sample_id'].map(split_lookup)

print('Total de muestras:', len(samples))
print('Balance por clase:')
display(samples['Flood'].value_counts().sort_index().rename(index={CLASS_NO_FLOOD: 'No inundado', CLASS_FLOOD: 'Inundado'}).to_frame('n'))
print('Distribución por partición y clase:')
display(pd.crosstab(samples['split'], samples['Flood']))
display(samples.head())


Total de muestras: 5000
Balance por clase:


,n
Flood,
No inundado,2500
Inundado,2500


Distribución por partición y clase:


Flood,0,1
split,,
test,500,500
train,1500,1500
validation,500,500


,sample_id,row,col,x,y_coord,Flood,split
0,S00001,2537,2395,435945.0,968385.0,0,train
1,S00002,553,1016,394575.0,1027905.0,1,train
2,S00003,853,1605,412245.0,1018905.0,1,validation
3,S00004,2164,644,383415.0,979575.0,0,test
4,S00005,1950,1376,405375.0,985995.0,1,train


## 6. Extraer las nueve variables de cada DEM y exportar CSV

In [ ]:
master_path = os.path.join(OUTPUT_DIR, 'ubicaciones_muestra_particiones.csv')
samples.to_csv(master_path, index=False)
print('Guardado:', master_path)

for dem_name, dem_path in DEM_PATHS.items():
    with rasterio.open(dem_path) as src:
        values = np.asarray(list(src.sample(coords, indexes=list(range(1, len(BAND_NAMES) + 1)))), dtype=np.float32)

    model_df = samples.copy()
    for i, band_name in enumerate(BAND_NAMES):
        model_df[band_name] = values[:, i]

    # Orden explícito: ID y ubicación, predictores, etiqueta y partición.
    columns = ['sample_id', 'row', 'col', 'x', 'y_coord'] + BAND_NAMES + ['Flood', 'split']
    model_df = model_df[columns]
    out_path = os.path.join(OUTPUT_DIR, f'muestras_{dem_name.lower()}_5000.csv')
    model_df.to_csv(out_path, index=False, float_format='%.6f')
    print(f'{dem_name}: {out_path} | filas={len(model_df)} | columnas={len(model_df.columns)}')

print('\\nExportación terminada.')


Guardado: /content/drive/MyDrive/Sinu_Flood_Susceptibility/04_samples/ubicaciones_muestra_particiones.csv
FABDEM: /content/drive/MyDrive/Sinu_Flood_Susceptibility/04_samples/muestras_fabdem_5000.csv | filas=5000 | columnas=16
Copernicus: /content/drive/MyDrive/Sinu_Flood_Susceptibility/04_samples/muestras_copernicus_5000.csv | filas=5000 | columnas=16
ANADEM: /content/drive/MyDrive/Sinu_Flood_Susceptibility/04_samples/muestras_anadem_5000.csv | filas=5000 | columnas=16
\nExportación terminada.


## 7. Verificación final de los archivos exportados

Comprueba que los tres CSV contienen las mismas muestras, etiquetas y particiones. Si algo falla, no continúes al entrenamiento.

In [ ]:
csv_paths = {
    name: os.path.join(OUTPUT_DIR, f'muestras_{name.lower()}_5000.csv')
    for name in DEM_PATHS
}
tables = {name: pd.read_csv(path) for name, path in csv_paths.items()}
base = tables['FABDEM'][['sample_id', 'row', 'col', 'Flood', 'split']]

for name, df in tables.items():
    assert len(df) == N_SAMPLES, f'{name}: número de filas incorrecto.'
    assert df['sample_id'].is_unique, f'{name}: sample_id repetidos.'
    assert df[['sample_id', 'row', 'col', 'Flood', 'split']].equals(base), f'{name}: muestras/etiquetas/particiones no coinciden con FABDEM.'
    assert df[BAND_NAMES].notna().all().all(), f'{name}: hay valores nulos en predictores.'

print('Correcto: los tres CSV tienen las mismas 5.000 ubicaciones, etiquetas y particiones.')
print('Archivos creados:')
for path in [master_path] + list(csv_paths.values()):
    print(path)



Correcto: los tres CSV tienen las mismas 5.000 ubicaciones, etiquetas y particiones.
Archivos creados:
/content/drive/MyDrive/Sinu_Flood_Susceptibility/04_samples/ubicaciones_muestra_particiones.csv
/content/drive/MyDrive/Sinu_Flood_Susceptibility/04_samples/muestras_fabdem_5000.csv
/content/drive/MyDrive/Sinu_Flood_Susceptibility/04_samples/muestras_copernicus_5000.csv
/content/drive/MyDrive/Sinu_Flood_Susceptibility/04_samples/muestras_anadem_5000.csv
Configuración inicial sugerida para la siguiente etapa: {'n_estimators': 300, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': None, 'random_state': 42, 'n_jobs': -1}
